In [2]:
# train_phobert_hatespeech.py
# Fine-tune PhoBERT for hate speech detection (multi-class)
# Train on train.csv, tune on dev.csv, evaluate once on test.csv

import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

# ===================== CONFIG =====================
TRAIN_PATH = "/home/uit2023/LuuTru/Thuchd/cs221/CS221_NLP_SA/UIT-ViHSD-preprocessed/train.csv"
DEV_PATH   = "/home/uit2023/LuuTru/Thuchd/cs221/CS221_NLP_SA/UIT-ViHSD-preprocessed/dev.csv"
TEST_PATH  = "/home/uit2023/LuuTru/Thuchd/cs221/CS221_NLP_SA/UIT-ViHSD-preprocessed/test.csv"

TEXT_COL  = "free_text"
LABEL_COL = "label_id"

MODEL_NAME = "vinai/phobert-base"   # hoặc "vinai/phobert-large"
OUTPUT_DIR = "./models/phobert_viHSD"
SAVE_INFO_PATH = os.path.join(OUTPUT_DIR, "final_info.json")

RANDOM_STATE = 42
SCORING = "f1_macro"

# Số trial random search (giảm nếu máy yếu/CPU)
N_TRIALS = 10

# ===================== UTIL =====================
def load_df(path: str) -> pd.DataFrame:
    df = pd.read_csv(path).dropna(subset=[TEXT_COL, LABEL_COL]).copy()
    df[TEXT_COL] = df[TEXT_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    return df

def overlap_count(a_texts, b_texts) -> int:
    return len(set(a_texts) & set(b_texts))

def eval_and_print(name, y_true, y_pred):
    print(f"\n===== {name} =====")
    print("f1_macro:", f1_score(y_true, y_pred, average="macro"))
    print("acc     :", accuracy_score(y_true, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
    print("\nReport:\n", classification_report(y_true, y_pred, digits=4))

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length: int):
        self.texts = list(texts)
        self.labels = list(labels) if labels is not None else None
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx: int):
        text = self.texts[idx]
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def make_compute_metrics():
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "f1_macro": f1_score(labels, preds, average="macro"),
            "accuracy": accuracy_score(labels, preds),
        }
    return compute_metrics

def train_one_trial(
    trial_id: int,
    model_name: str,
    num_labels: int,
    tokenizer,
    train_texts, train_labels,
    dev_texts, dev_labels,
    hp: dict,
):
    # fresh model each trial
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=num_labels,
    )

    train_ds = TextDataset(train_texts, train_labels, tokenizer, max_length=hp["max_length"])
    dev_ds   = TextDataset(dev_texts,   dev_labels,   tokenizer, max_length=hp["max_length"])

    out_dir = os.path.join(OUTPUT_DIR, f"_trial_{trial_id}")
    args = TrainingArguments(
        output_dir=out_dir,
        learning_rate=hp["learning_rate"],
        per_device_train_batch_size=hp["batch_size"],
        per_device_eval_batch_size=hp["batch_size"],
        num_train_epochs=hp["epochs"],
        weight_decay=hp["weight_decay"],
        warmup_ratio=hp["warmup_ratio"],
        eval_strategy="epoch",
        save_strategy="no",            # không save trong lúc tune để đỡ tốn disk
        logging_strategy="steps",
        logging_steps=50,
        report_to="none",
        seed=RANDOM_STATE,
        data_seed=RANDOM_STATE,
        fp16=torch.cuda.is_available(), # chỉ bật fp16 nếu có GPU
        gradient_accumulation_steps=hp["grad_accum"],
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=dev_ds,
        tokenizer=tokenizer,
        compute_metrics=make_compute_metrics(),
    )

    trainer.train()
    metrics = trainer.evaluate()

    # predict on dev for detailed report
    pred_out = trainer.predict(dev_ds)
    dev_preds = np.argmax(pred_out.predictions, axis=-1)

    return {
        "metrics": metrics,
        "dev_preds": dev_preds,
        "hp": hp,
    }

def sample_hyperparams(rng: random.Random) -> dict:
    # space hyperparam đơn giản, phù hợp PhoBERT
    # (bạn có thể mở rộng nếu muốn)
    lr_choices = [1e-5, 2e-5, 3e-5, 5e-5]
    bs_choices = [8, 16]
    ep_choices = [2, 3, 4]
    wd_choices = [0.0, 0.01]
    wr_choices = [0.0, 0.06, 0.1]
    ml_choices = [128, 256]

    hp = {
        "learning_rate": rng.choice(lr_choices),
        "batch_size": rng.choice(bs_choices),
        "epochs": rng.choice(ep_choices),
        "weight_decay": rng.choice(wd_choices),
        "warmup_ratio": rng.choice(wr_choices),
        "max_length": rng.choice(ml_choices),
        # nếu batch size nhỏ mà GPU khỏe có thể để 1
        "grad_accum": rng.choice([1, 2]),
    }
    return hp

# ===================== MAIN =====================
def main():
    set_seed(RANDOM_STATE)
    rng = random.Random(RANDOM_STATE)

    df_train = load_df(TRAIN_PATH)
    df_dev   = load_df(DEV_PATH)
    df_test  = load_df(TEST_PATH)

    X_train = df_train[TEXT_COL].tolist()
    y_train = df_train[LABEL_COL].tolist()

    X_dev   = df_dev[TEXT_COL].tolist()
    y_dev   = df_dev[LABEL_COL].tolist()

    X_test  = df_test[TEXT_COL].tolist()
    y_test  = df_test[LABEL_COL].tolist()

    print("Overlap train-dev:", overlap_count(X_train, X_dev))
    print("Overlap train-test:", overlap_count(X_train, X_test))
    print("Overlap dev-test:", overlap_count(X_dev, X_test))

    # infer num labels
    all_labels = sorted(set(y_train) | set(y_dev) | set(y_test))
    num_labels = len(all_labels)
    print("num_labels =", num_labels, "| labels =", all_labels)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

    # ===================== TUNE on DEV =====================
    best = None
    tried = set()

    for t in range(N_TRIALS):
        hp = sample_hyperparams(rng)
        key = tuple(sorted(hp.items()))
        if key in tried:
            continue
        tried.add(key)

        print(f"\n================ TRIAL {t+1}/{N_TRIALS} ================")
        print("HP:", hp)

        result = train_one_trial(
            trial_id=t,
            model_name=MODEL_NAME,
            num_labels=num_labels,
            tokenizer=tokenizer,
            train_texts=X_train,
            train_labels=y_train,
            dev_texts=X_dev,
            dev_labels=y_dev,
            hp=hp,
        )

        dev_f1 = float(result["metrics"].get("eval_f1_macro", float("nan")))
        dev_acc = float(result["metrics"].get("eval_accuracy", float("nan")))
        print(f"TRIAL dev_f1_macro={dev_f1:.6f} | dev_acc={dev_acc:.6f}")

        if (best is None) or (dev_f1 > best["dev_f1_macro"]):
            best = {
                "dev_f1_macro": dev_f1,
                "dev_acc": dev_acc,
                "hp": hp,
                "dev_preds": result["dev_preds"],
            }

    if best is None:
        raise RuntimeError("Không tìm được trial hợp lệ (có thể do lỗi môi trường/thiếu GPU/thiếu package).")

    print("\n===== BEST BY DEV =====")
    print("best_dev_f1_macro:", best["dev_f1_macro"])
    print("best_hp:", best["hp"])

    # report best trial on DEV (train-only fit in that trial)
    eval_and_print("DEV (best trial)", y_dev, best["dev_preds"])

    # ===================== FINAL: Train from scratch on TRAIN only =====================
    print("\n===== FINAL TRAIN (TRAIN only) -> TEST (1 lần) =====")
    final_hp = best["hp"]

    final_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels
    )

    train_ds = TextDataset(X_train, y_train, tokenizer, max_length=final_hp["max_length"])
    dev_ds   = TextDataset(X_dev,   y_dev,   tokenizer, max_length=final_hp["max_length"])
    test_ds  = TextDataset(X_test,  y_test,  tokenizer, max_length=final_hp["max_length"])

    final_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        learning_rate=final_hp["learning_rate"],
        per_device_train_batch_size=final_hp["batch_size"],
        per_device_eval_batch_size=final_hp["batch_size"],
        num_train_epochs=final_hp["epochs"],
        weight_decay=final_hp["weight_decay"],
        warmup_ratio=final_hp["warmup_ratio"],
        eval_strategy="epoch",
        save_strategy="epoch",         # save để lưu model cuối
        load_best_model_at_end=False,  # tránh “dính” dev
        logging_strategy="steps",
        logging_steps=50,
        report_to="none",
        seed=RANDOM_STATE,
        data_seed=RANDOM_STATE,
        fp16=torch.cuda.is_available(),
        gradient_accumulation_steps=final_hp["grad_accum"],
    )

    final_trainer = Trainer(
        model=final_model,
        args=final_args,
        train_dataset=train_ds,
        eval_dataset=dev_ds,   # vẫn eval cho log thôi, KHÔNG dùng để chọn model
        tokenizer=tokenizer,
        compute_metrics=make_compute_metrics(),
    )

    final_trainer.train()

    # DEV report (optional)
    dev_out = final_trainer.predict(dev_ds)
    dev_pred = np.argmax(dev_out.predictions, axis=-1)
    eval_and_print("DEV (final train-only)", y_dev, dev_pred)

    # TEST (1 lần)
    test_out = final_trainer.predict(test_ds)
    test_pred = np.argmax(test_out.predictions, axis=-1)
    eval_and_print("TEST (final train-only)", y_test, test_pred)

    # ===================== SAVE =====================
    # Save final model + tokenizer
    final_save_dir = os.path.join(OUTPUT_DIR, "final_model")
    Path(final_save_dir).mkdir(parents=True, exist_ok=True)
    final_trainer.model.save_pretrained(final_save_dir)
    tokenizer.save_pretrained(final_save_dir)

    info = {
        "model_name": MODEL_NAME,
        "num_labels": num_labels,
        "best_dev_f1_macro": float(best["dev_f1_macro"]),
        "best_dev_acc": float(best["dev_acc"]),
        "best_hyperparams": final_hp,
        "final_dev_f1_macro": float(f1_score(y_dev, dev_pred, average="macro")),
        "final_dev_acc": float(accuracy_score(y_dev, dev_pred)),
        "test_f1_macro": float(f1_score(y_test, test_pred, average="macro")),
        "test_acc": float(accuracy_score(y_test, test_pred)),
        "train_path": TRAIN_PATH,
        "dev_path": DEV_PATH,
        "test_path": TEST_PATH,
    }
    with open(SAVE_INFO_PATH, "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)

    print(f"\nSaved final model to: {final_save_dir}")
    print(f"Saved info to      : {SAVE_INFO_PATH}")

if __name__ == "__main__":
    main()


Overlap train-dev: 381
Overlap train-test: 897
Overlap dev-test: 134
num_labels = 3 | labels = [0, 1, 2]

================ TRIAL 1/10 ================
HP: {'learning_rate': 1e-05, 'batch_size': 8, 'epochs': 4, 'weight_decay': 0.01, 'warmup_ratio': 0.0, 'max_length': 128, 'grad_accum': 1}


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_854972/2716431350.py:131: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/home/uit2023/miniconda3/envs/downloadData/lib/python3.14/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,F1 Macro,Accuracy
1,0.419400,0.413440,0.563150,0.849551


/home/uit2023/miniconda3/envs/downloadData/lib/python3.14/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


KeyboardInterrupt: 